In [1]:
!pip install transformers torch accelerate flask-ngrok pyngrok flask

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "alotanna/llama2-7b-ghana-climate"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
print(f"Device: {model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/3.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Device: cuda:0


In [ ]:
# Create Flask API Server

from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint"""
    return jsonify({"status": "healthy", "model": model_name})

@app.route('/generate', methods=['POST'])
def generate():
    """
    Generate text from prompt

    Expected JSON:
    {
        "prompt": "Your prompt here",
        "max_new_tokens": 512,
        "temperature": 0.3,
        "top_p": 0.85,
        "repetition_penalty": 1.15
    }
    """
    try:
        data = request.json

        # Get parameters
        prompt = data.get('prompt', '')
        max_new_tokens = data.get('max_new_tokens', 512)
        temperature = data.get('temperature', 0.3)
        top_p = data.get('top_p', 0.85)
        repetition_penalty = data.get('repetition_penalty', 1.15)

        if not prompt:
            return jsonify({"error": "No prompt provided"}), 400

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs['input_ids'].shape[1]

        # Generate
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                do_sample=True,
                use_cache = False,
                pad_token_id=tokenizer.pad_token_id,
                no_repeat_ngram_size=3
            )

        # Decode only generated tokens
        generated_ids = output[0][input_length:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        return jsonify({
            "generated_text": generated_text.strip(),
            "input_length": input_length,
            "output_length": len(generated_ids)
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

In [ ]:
ngrok.set_auth_token("2jKJALyFxN5nUdqaL8ZqIYmeDbT_4AGfruz5H6NGtFX9BcShR")

STATIC_DOMAIN = "miquel-nonintersecting-pachydermatously.ngrok-free.dev"

# Connect with your static domain
public_url = ngrok.connect(5000, domain=STATIC_DOMAIN)

# Start ngrok tunnel
#public_url = ngrok.connect(5000)
print("API SERVER IS RUNNING!")
print(f"Public URL: {public_url}")
print("\nEndpoints:")
print(f"  Health check: {public_url}/health")
print(f"  Generate: {public_url}/generate (POST)")
# Run Flask app
app.run(port=5000)

API SERVER IS RUNNING!
Public URL: NgrokTunnel: "https://miquel-nonintersecting-pachydermatously.ngrok-free.dev" -> "http://localhost:5000"

Endpoints:
  Health check: NgrokTunnel: "https://miquel-nonintersecting-pachydermatously.ngrok-free.dev" -> "http://localhost:5000"/health
  Generate: NgrokTunnel: "https://miquel-nonintersecting-pachydermatously.ngrok-free.dev" -> "http://localhost:5000"/generate (POST)
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:03] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:10] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:16] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:23] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:28] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:39] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:45] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Dec/2025 23:38:53] "POST /generate HTTP/1.1" 200 -
